In [162]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep csv = "1.1"
:dep ndarray = { version = "0.16", features = ["rayon"] }
:dep linfa = { version = "0.8.1" }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"
:dep linfa-preprocessing =  "0.8.1" 
:dep linfa-elasticnet = { version =  "0.8.1" }
:dep linfa-reduction =  "0.8.1" 
:dep polars = { version = "0.53", features = ["ndarray"] }
:dep polars-core = "0.53"
:dep image = "0.24"
:opt 3

Optimization: 3


In [157]:
use smartcore::{neighbors::knn_classifier::*,linalg::basic::matrix::DenseMatrix,
                naive_bayes::gaussian::GaussianNB,tree::decision_tree_classifier::DecisionTreeClassifier,
                metrics::distance::Distances,tree::decision_tree_classifier::DecisionTreeClassifierParameters,
                tree::decision_tree_classifier::SplitCriterion};
use csv;
use linfa::{Dataset};
use ndarray::{Array1, Array2, ArrayView1, ArrayView2, Axis, Ix1, array,s};
use rand::SeedableRng;
use rand_chacha::ChaCha12Rng;
use linfa::prelude::*;
use linfa_reduction::Pca;
use plotters::prelude::*;
use ndarray::Array2;
use std::path::Path as StdPath;
use linfa_preprocessing::linear_scaling::LinearScaler;
use plotters::prelude::*;
use std::path::Path as StdPath;
use polars_core::prelude::*;
use ndarray::Dim;
use ndarray::ArrayBase;
use ndarray::OwnedRepr;
use polars::prelude::CsvReadOptions;
use polars::prelude::SerReader;

In [156]:
let path = "DroneTelemetry.csv";
let mut reader = csv::Reader::from_path(path).unwrap();
let mut records: Array2<f64> = Array2::zeros((0,7));
let mut targets: Array1<f64> = Array1::zeros(0);
let mut rng = ChaCha12Rng::seed_from_u64(5);
//предсказание расхода батареи
//в X нужны столбцы 5 6 7 10 11 12 18
//в Y нужен 16

In [148]:
let mut first_string=false;
for data in reader.records(){
        let mut parse_err = false;
        let rec = data.unwrap();
        let rec5:f64 = match rec[5].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec6:f64 = match rec[6].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec7:f64 = match rec[7].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec10:f64 = match rec[10].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec11:f64 = match rec[11].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec12:f64 = match rec[12].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec17:f64 = match rec[17].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        let rec15:f64 = match rec[15].parse::<f64>(){
            Ok(num) => {num},
            Err(_) => {parse_err=true;
                        continue},
        };
        if parse_err==false{
            let array = array![[
                              rec5,rec6,rec7,rec10,rec11,rec12,rec17
                                ]];
            let _  = records.append(Axis(0), array.view());
            let target_array = array![rec15];
            let _  = targets.append(Axis(0),target_array.view());
        }
    }
let dataset= Dataset::new(records, targets);
let (train_data,test_data) = dataset.shuffle(&mut rng)
                                     .split_with_ratio(0.7);

In [149]:
let x = DenseMatrix::from_2d_vec(
                              &train_data.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let y = train_data.targets()
                .map(|value| *value as i32)
                .to_vec();
let knn=KNNClassifier::fit(&x, &y, Default::default()).unwrap(); //обучение

In [150]:
let test_x = DenseMatrix::from_2d_vec(
                              &test_data.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let test_y = test_data.targets()
                .map(|value| *value as i32)
                .to_vec();
let knn_params = KNNClassifierParameters::default()
                                    .with_k(2)
                                    .with_distance(Distances::manhattan());
let knn=KNNClassifier::fit(&x, &y, knn_params).unwrap(); 
let dt_params = DecisionTreeClassifierParameters::default()
                                                  .with_criterion(SplitCriterion::Entropy)
                                                  .with_max_depth(25)
                                                  .with_min_samples_leaf(1)
                                                  .with_min_samples_split(1);
let dt = DecisionTreeClassifier::fit(&x, &y, dt_params).unwrap(); 

In [151]:
let test_x = DenseMatrix::from_2d_vec(
                              &test_data.records().axis_iter(Axis(0))
                                                 .map(|row| row.to_vec())
                                                 .collect::<Vec<Vec<f64>>>()
                                );
let test_y = test_data.targets()
                .map(|value| *value as i32)
                .to_vec();
let knn_res = knn.predict(&test_x).unwrap();
let dt_res = dt.predict(&test_x).unwrap();

In [152]:
let mut knn_acc = 0.0;
let mut gnb_acc = 0.0;
let mut dt_acc = 0.0;
for i in 0..test_y.len(){
    if knn_res[i] == test_y[i]{
        knn_acc=knn_acc+1.0;
    }
    if dt_res[i] == test_y[i]{
        dt_acc=dt_acc+1.0;
    }
}
knn_acc=knn_acc/ (test_y.len() as f64);
dt_acc=dt_acc/ (test_y.len() as f64);
println!("knn accuracity: {}  | dt accuracity {}",knn_acc,dt_acc);

knn accuracity: 0.6402439024390244  | dt accuracity 0.6951219512195121


In [158]:
let mut df = CsvReadOptions::default()
        .with_has_header(false)
        .try_into_reader_with_file_path(Some("DroneTelemetry.csv".into())).unwrap()
        .finish().unwrap();

In [160]:
println!("{:?}",df)

shape: (554, 22)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ column_1 ┆ column_2  ┆ column_3  ┆ column_4  ┆ … ┆ column_19 ┆ column_20 ┆ column_21 ┆ column_22 │
│ ---      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str      ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Drone ID ┆ Applicati ┆ Drone     ┆ Drone     ┆ … ┆ Obstacles ┆ Flight    ┆ Regulator ┆ Notes     │
│          ┆ on        ┆ Size      ┆ Model     ┆   ┆ Encounter ┆ Status    ┆ y         ┆           │
│          ┆           ┆           ┆           ┆   ┆ ed        ┆           ┆ Approval  ┆           │
│          ┆           ┆           ┆           ┆   ┆           ┆           ┆ ID        ┆           │
│ D001     ┆ Package   ┆ Medium    ┆ FlyHigh   ┆ … ┆ No        ┆ Completed

()

In [168]:
let mut df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("DroneTelemetry.csv".into())).unwrap()
    .finish().unwrap();
let numeric_cols = [
    "Propeller Count",
    "Max Carry Weight (kg)",
    "Actual Carry Weight (kg)",
    "Altitude (meters)",
    "Flight Duration (minutes)",
    "Distance Flown (km)",
    "Battery Remaining (%)",
    "GPS Accuracy (meters)",
    "Wind Speed (m/s)"
];

let mut numeric_data = Vec::new();
let mut drone_ids = Vec::new();

for row_idx in 0..df.height() {
    let drone_id_val = df.column("Drone ID").unwrap().get(row_idx);
    let drone_id = match drone_id_val {
        Ok(val) => val.to_string(),
        Err(_) => "unknown".to_string()
    };
    
    let mut row_values = Vec::new();
    let mut has_numeric = true;
    
    for col_name in &numeric_cols {
        if let Ok(series) = df.column(col_name) {
            match series.get(row_idx) {
                Ok(val) => {
                    let val_str = val.to_string();
                    if let Ok(num) = val_str.parse::<f64>() {
                        row_values.push(num);
                    } else {
                        has_numeric = false;
                        break;
                    }
                },
                Err(_) => {
                    has_numeric = false;
                    break;
                }
            }
        }
    }
    
    if has_numeric && !row_values.is_empty() {
        numeric_data.push(row_values);
        drone_ids.push(drone_id);
    }
}

let shape = (numeric_data.len(), numeric_data[0].len());
let x_vec: Vec<f64> = numeric_data.into_iter().flatten().collect();
let x_final = ndarray::Array2::from_shape_vec(shape, x_vec).unwrap();
let y_final = ndarray::Array1::from_iter(0..shape.0);
let dataset = Dataset::new(x_final, y_final);

let scaler: LinearScaler<f64> = LinearScaler::standard().fit(&dataset).expect("scale e");
let dataset = scaler.transform(dataset);

let pca: Pca<f64> = Pca::params(2).fit(&dataset).unwrap();
let records = pca.predict(&dataset);

let root = BitMapBackend::new("drone_pca_clusters.png", (4000, 4000)).into_drawing_area();
root.fill(&WHITE).unwrap();

let x_min = records.column(0).iter().fold(f64::INFINITY, |a, &b| a.min(b));
let x_max = records.column(0).iter().fold(f64::NEG_INFINITY, |a, &b| a.max(b));
let y_min = records.column(1).iter().fold(f64::INFINITY, |a, &b| a.min(b));
let y_max = records.column(1).iter().fold(f64::NEG_INFINITY, |a, &b| a.max(b));

let x_padding = (x_max - x_min) * 0.1;
let y_padding = (y_max - y_min) * 0.1;
let x_range = (x_min - x_padding)..(x_max + x_padding);
let y_range = (y_min - y_padding)..(y_max + y_padding);

let mut chart = ChartBuilder::on(&root)
    .caption("PCA", ("sans-serif", 40))
    .margin(30)
    .x_label_area_size(40)
    .y_label_area_size(40)
    .build_cartesian_2d(x_range, y_range).unwrap();
let colors = [RED, BLUE, GREEN, CYAN, MAGENTA, YELLOW, RGBColor(255, 165, 0), RGBColor(128, 0, 128)];

for id in 0..drone_ids.len() {
    let row = records.row(id);
    let color = colors[id % colors.len()];
    
    chart.draw_series(std::iter::once(
        Circle::new((row[0], row[1]), 5, color.filled())
    )).unwrap();
    
    chart.draw_series(std::iter::once(
        Text::new(
            drone_ids[id].clone(),
            (row[0] + 0.05, row[1] + 0.05),
            ("sans-serif", 12).into_font()
        )
    )).unwrap();
}

root.present().unwrap();

let img = image::open("drone_pca_clusters.png").expect("open err");
let scaled = img.resize(1000, 1000, image::imageops::FilterType::Lanczos3);
scaled.save("drone_pca_clusters_scaled.png").expect("save err");